# COALESCE, ISNULL, NULLIF

In [1]:
import pandas as pd
import numpy as np

In [2]:
df_customers = pd.read_csv('data/sales_customers.csv')
df_employees = pd.read_csv('data/sales_employees.csv')
df_orders = pd.read_csv('data/sales_orders.csv')
df_orderarchive = pd.read_csv('data/sales_ordersarchive.csv')
df_products = pd.read_csv('data/sales_products.csv')

### ISNULL()

#### Replace Null with some Value

### COALESCE

#### Returns the first non-null value from a list

### ISNULL() | COALESCE

#### Handle the NULL before doing data aggregations

### SQL TASK

#### Find the average scores of the customers

```SQL
SELECT
    AVG(score) AS avg_score, -- With NULL
    AVG(COALESCE(score, 0)) AS avg_score -- Replace NULL with 0
FROM sales.customers;
```

In [12]:
avg_score_with_null = df_customers['score'].mean()
avg_score_with_no_null = df_customers['score'].fillna(0).mean()

df_score_avg = pd.DataFrame({
    'avg_score_ignoring_null': [avg_score_with_null],
    'avg_score_with_zeros': [avg_score_with_no_null]
})

df_score_avg

,avg_score_ignoring_null,avg_score_with_zeros
0,625.0,500.0


### SQL TASK
#### Display the full name of customers in a single field
#### By merging their first and last names,
#### and add 10 bonus points to each customer's score

```SQL
SELECT
    COALESCE(firstname, ' ') || COALESCE(lastname, ' ') AS fullname,
    COALESCE(score, 0) + 10 AS score
FROM sales.customers;
```

In [20]:
df_res = df_customers.copy()

df_res['fullname'] = (
    df_res['firstname'].fillna('') + ' ' + df_res['lastname'].fillna('')
)

df_res['score_plus_10'] = df_res['score'].fillna(0) + 10

df_res[['fullname','score_plus_10']]



,fullname,score_plus_10
0,Jossef Goldberg,360.0
1,Kevin Brown,910.0
2,Mary,760.0
3,Mark Schwarz,510.0
4,Anna Adams,10.0


### ISNULL | COALESCE -- USE CASE --

#### Handle the NULL before JOINING tables

### Handle the NULL before sorting data

### SQL TASK

#### Sort the customers from lowest to hihest scores, with nulls appearing last

```SQL
SELECT
    score,
    CASE WHEN score IS NULL THEN 1 ELSE 0 END AS Flag
FROM sales.customers
ORDER BY Flag ASC, score ASC;
```

In [22]:
df_res = df_customers.copy()

df_res['flag'] = df_res['score'].isna().astype(int)

df_res = df_res.sort_values(by=['flag','score'], ascending=[True,True])

df_res[['score','flag']]

,score,flag
0,350.0,0
3,500.0,0
2,750.0,0
1,900.0,0
4,NaN,1


### NULLIF()

#### Compares two expressions returns:
- NULL, if they are equal
- First Value, if they are not equal

### SQL TASK
#### Find the sales price for each order by dividing the sales by the quantity

```SQL
SELECT
    orderid,
    sales,
    quantity,
    CASE WHEN quantity = 0 THEN NULL ELSE (sales/quantity) END AS sales_price,
    sales / NULLIF(quantity,0) AS price
FROM sales.orders;
```

In [24]:
df_orders['price'] = df_orders['sales'] / df_orders['quantity'].replace(0, np.nan)

df_orders['price']


0    10.0
1    15.0
2    10.0
3    30.0
4    25.0
5    25.0
6    15.0
7    30.0
8    10.0
9     NaN
Name: price, dtype: float64

### SQL TASK

#### Identify the customers who have no scores

```SQL
SELECT
    *
FROM sales.customers
WHERE score IS NULL;
```

In [31]:
df_null_score = df_customers.copy()

res1 = df_null_score[df_null_score['score'].isna()]

res2 = res = df_null_score[df_null_score['score'].notnull()]

res2

,customerid,firstname,lastname,country,score
0,1,Jossef,Goldberg,Germany,350.0
1,2,Kevin,Brown,USA,900.0
2,3,Mary,NaN,USA,750.0
3,4,Mark,Schwarz,Germany,500.0


### SQL TASK

#### List all details for customers who have not placed any orders

```SQL
SELECT
    c.*,
    o.orderdate
FROM sales.customers c
LEFT JOIN sales.orders o ON c.customerid = o.customerid
WHERE o.customerid IS NULL;
```

In [36]:
df_merged = pd.merge(
    df_customers,
    df_orders[['customerid','orderdate']],
    on = 'customerid',
    how = 'left',
    indicator = True
)

result = df_merged[df_merged['_merge'] == 'left_only'].drop(columns=['_merge'])

result

,customerid,firstname,lastname,country,score,orderdate
10,5,Anna,Adams,USA,NaN,NaN
